# Baseline Modeling and Pipeline Simulation

This notebook covers:
1. **Baseline Modeling**: Training and evaluating models on the ULB MLG credit card fraud dataset
2. **Pipeline Simulation**: Transitioning to real-time fraud detection pipeline using fraudTrain/fraudTest datasets


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## Part 1: Baseline Modeling with ULB MLG Dataset

We'll start by training baseline models on the creditcard.csv dataset (ULB MLG).

**Note:** This section uses the preprocessed data from Notebook 02 (Feature Engineering), which includes:
- Scaled features using the best-performing scaler (determined in Notebook 02)
- Train/test split with stratification
- Multiple resampling methods (RUS, ROS, SMOTE) for comparison


In [ ]:
# Load preprocessed data from Notebook 02 (Feature Engineering)
data_dir = Path('../data/processed')

# Check if processed data exists from Notebook 02
if data_dir.exists() and (data_dir / 'X_train_scaled.csv').exists():
    print("="*70)
    print("LOADING PREPROCESSED DATA FROM NOTEBOOK 02")
    print("="*70)
    
    # Load scaled data
    X_train_ulb = pd.read_csv(data_dir / 'X_train_scaled.csv')
    X_test_ulb = pd.read_csv(data_dir / 'X_test_scaled.csv')
    y_train_ulb = pd.read_csv(data_dir / 'y_train.csv').squeeze()
    y_test_ulb = pd.read_csv(data_dir / 'y_test.csv').squeeze()
    
    # Load scaler metadata
    if (data_dir / 'scaler_metadata.pkl').exists():
        with open(data_dir / 'scaler_metadata.pkl', 'rb') as f:
            scaler_metadata = pickle.load(f)
        scaler_ulb = scaler_metadata['scaler']
        print(f"\n✓ Using {scaler_metadata['scaler_name']} from Notebook 02")
        print(f"  Features: {len(scaler_metadata['feature_names'])}")
    else:
        # Fallback to loading scaler directly
        with open(data_dir / 'scaler.pkl', 'rb') as f:
            scaler_ulb = pickle.load(f)
        print(f"\n✓ Loaded scaler from Notebook 02")
    
    # Convert scaled data to DataFrame if needed (already should be)
    if not isinstance(X_train_ulb, pd.DataFrame):
        X_train_ulb = pd.DataFrame(X_train_ulb, columns=scaler_metadata['feature_names'])
        X_test_ulb = pd.DataFrame(X_test_ulb, columns=scaler_metadata['feature_names'])
    
    print(f"\n✓ Loaded preprocessed data from Notebook 02")
    print(f"  Training shape: {X_train_ulb.shape}")
    print(f"  Test shape: {X_test_ulb.shape}")
    print(f"  Training fraud rate: {y_train_ulb.mean() * 100:.2f}%")
    print(f"  Test fraud rate: {y_test_ulb.mean() * 100:.2f}%")
    
else:
    print("⚠ Processed data from Notebook 02 not found.")
    print("  Falling back to raw data processing...")
    print("  (This is not recommended - run Notebook 02 first)")
    
    # Fallback: Load and process raw data
    df_ulb = pd.read_csv('../data/creditcard.csv')
    X_ulb = df_ulb.drop('Class', axis=1)
    y_ulb = df_ulb['Class']
    
    X_train_ulb, X_test_ulb, y_train_ulb, y_test_ulb = train_test_split(
        X_ulb, y_ulb, test_size=0.2, random_state=42, stratify=y_ulb
    )
    
    scaler_ulb = RobustScaler()
    X_train_ulb = pd.DataFrame(
        scaler_ulb.fit_transform(X_train_ulb), 
        columns=X_train_ulb.columns
    )
    X_test_ulb = pd.DataFrame(
        scaler_ulb.transform(X_test_ulb), 
        columns=X_test_ulb.columns
    )


In [ ]:
# Data is already loaded and preprocessed from Notebook 02
# No additional preprocessing needed here
print("Data ready for modeling!")


In [ ]:
# Data is already scaled from Notebook 02
# Use the same variable names for consistency
X_train_ulb_scaled = X_train_ulb.copy()
X_test_ulb_scaled = X_test_ulb.copy()

print("✓ Using pre-scaled data from Notebook 02")


### Baseline Model 1: Logistic Regression


In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(
    class_weight='balanced',  # Handle class imbalance
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_ulb_scaled, y_train_ulb)

# Predictions
y_pred_lr = lr_model.predict(X_test_ulb_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_ulb_scaled)[:, 1]

# Evaluation
print("Logistic Regression Results:")
print(f"ROC-AUC Score: {roc_auc_score(y_test_ulb, y_pred_proba_lr):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_ulb, y_pred_lr))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_ulb, y_pred_lr))


### Baseline Model 2: Random Forest


In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_ulb_scaled, y_train_ulb)

# Predictions
y_pred_rf = rf_model.predict(X_test_ulb_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_ulb_scaled)[:, 1]

# Evaluation
print("Random Forest Results:")
print(f"ROC-AUC Score: {roc_auc_score(y_test_ulb, y_pred_proba_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_ulb, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_ulb, y_pred_rf))


In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': X_train_ulb_scaled.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize feature importance
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


### Resampling Methods Comparison

Now we'll compare different resampling methods prepared in Notebook 02 to determine which works best for our models.


In [ ]:
# Load resampled datasets from Notebook 02
resampling_methods = {}

# Check if resampled datasets exist
if (data_dir / 'X_train_rus.csv').exists():
    resampling_methods['RUS'] = {
        'X_train': pd.read_csv(data_dir / 'X_train_rus.csv'),
        'y_train': pd.read_csv(data_dir / 'y_train_rus.csv').squeeze(),
        'use_class_weight': False
    }
    
if (data_dir / 'X_train_ros.csv').exists():
    resampling_methods['ROS'] = {
        'X_train': pd.read_csv(data_dir / 'X_train_ros.csv'),
        'y_train': pd.read_csv(data_dir / 'y_train_ros.csv').squeeze(),
        'use_class_weight': False
    }
    
if (data_dir / 'X_train_smote.csv').exists():
    resampling_methods['SMOTE'] = {
        'X_train': pd.read_csv(data_dir / 'X_train_smote.csv'),
        'y_train': pd.read_csv(data_dir / 'y_train_smote.csv').squeeze(),
        'use_class_weight': False
    }

# Add original dataset with class_weight
resampling_methods['Original (class_weight)'] = {
    'X_train': X_train_ulb_scaled,
    'y_train': y_train_ulb,
    'use_class_weight': True
}

print("="*70)
print("RESAMPLING METHODS COMPARISON")
print("="*70)
print(f"\nAvailable methods: {list(resampling_methods.keys())}")
for method_name, data in resampling_methods.items():
    print(f"\n{method_name}:")
    print(f"  Training samples: {len(data['y_train']):,}")
    print(f"  Class distribution: {data['y_train'].value_counts().to_dict()}")
    
# Only proceed if we have resampled datasets
if len(resampling_methods) > 1:
    print("\n✓ Resampled datasets loaded. Proceeding with comparison...")
else:
    print("\n⚠ Resampled datasets not found. Skipping resampling comparison.")
    print("  Run Notebook 02 to generate resampled datasets.")


In [ ]:
# Compare models across different resampling methods
if len(resampling_methods) > 1:
    comparison_results = []
    
    print("\nTraining and evaluating models on different resampling methods...")
    print("="*70)
    
    for method_name, data in resampling_methods.items():
        print(f"\n{method_name}...")
        
        # Train Logistic Regression
        lr = LogisticRegression(
            class_weight='balanced' if data['use_class_weight'] else None,
            max_iter=1000,
            random_state=42,
            n_jobs=-1
        )
        lr.fit(data['X_train'], data['y_train'])
        y_pred_proba_lr = lr.predict_proba(X_test_ulb_scaled)[:, 1]
        roc_auc_lr = roc_auc_score(y_test_ulb, y_pred_proba_lr)
        
        # Train Random Forest
        rf = RandomForestClassifier(
            n_estimators=100,
            class_weight='balanced' if data['use_class_weight'] else None,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(data['X_train'], data['y_train'])
        y_pred_proba_rf = rf.predict_proba(X_test_ulb_scaled)[:, 1]
        roc_auc_rf = roc_auc_score(y_test_ulb, y_pred_proba_rf)
        
        comparison_results.append({
            'Method': method_name,
            'LR_ROC_AUC': roc_auc_lr,
            'RF_ROC_AUC': roc_auc_rf
        })
        
        print(f"  LR ROC-AUC: {roc_auc_lr:.4f}")
        print(f"  RF ROC-AUC: {roc_auc_rf:.4f}")
    
    # Create comparison DataFrame
    comparison_df = pd.DataFrame(comparison_results)
    print("\n" + "="*70)
    print("RESAMPLING METHODS COMPARISON RESULTS")
    print("="*70)
    print(comparison_df.to_string(index=False))
else:
    print("\n⚠ Skipping resampling comparison - datasets not available.")


In [ ]:
# Visualize resampling comparison
if 'comparison_df' in locals() and len(comparison_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # ROC-AUC comparison
    x = np.arange(len(comparison_df))
    width = 0.35
    
    axes[0].bar(x - width/2, comparison_df['LR_ROC_AUC'], width, 
                label='Logistic Regression', alpha=0.8, color='blue')
    axes[0].bar(x + width/2, comparison_df['RF_ROC_AUC'], width, 
                label='Random Forest', alpha=0.8, color='orange')
    axes[0].set_xlabel('Resampling Method')
    axes[0].set_ylabel('ROC-AUC Score')
    axes[0].set_title('Model Performance by Resampling Method', fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(comparison_df['Method'], rotation=45, ha='right')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_ylim([comparison_df[['LR_ROC_AUC', 'RF_ROC_AUC']].min().min() - 0.01,
                      comparison_df[['LR_ROC_AUC', 'RF_ROC_AUC']].max().max() + 0.01])
    
    # Add value labels
    for i, row in comparison_df.iterrows():
        axes[0].text(i - width/2, row['LR_ROC_AUC'], f"{row['LR_ROC_AUC']:.3f}",
                    ha='center', va='bottom', fontsize=8)
        axes[0].text(i + width/2, row['RF_ROC_AUC'], f"{row['RF_ROC_AUC']:.3f}",
                    ha='center', va='bottom', fontsize=8)
    
    # Best method identification
    best_lr_idx = comparison_df['LR_ROC_AUC'].idxmax()
    best_rf_idx = comparison_df['RF_ROC_AUC'].idxmax()
    
    axes[1].text(0.1, 0.8, f"Best LR Method:\n{comparison_df.loc[best_lr_idx, 'Method']}\n"
                           f"ROC-AUC: {comparison_df.loc[best_lr_idx, 'LR_ROC_AUC']:.4f}",
                 transform=axes[1].transAxes, fontsize=12, fontweight='bold',
                 bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    axes[1].text(0.1, 0.5, f"Best RF Method:\n{comparison_df.loc[best_rf_idx, 'Method']}\n"
                           f"ROC-AUC: {comparison_df.loc[best_rf_idx, 'RF_ROC_AUC']:.4f}",
                 transform=axes[1].transAxes, fontsize=12, fontweight='bold',
                 bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))
    axes[1].axis('off')
    axes[1].set_title('Best Performing Methods', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Store best methods for later use
    best_method_lr = comparison_df.loc[best_lr_idx, 'Method']
    best_method_rf = comparison_df.loc[best_rf_idx, 'Method']
    print(f"\n✓ Best method for Logistic Regression: {best_method_lr}")
    print(f"✓ Best method for Random Forest: {best_method_rf}")
else:
    print("\n⚠ Skipping visualization - comparison data not available.")


In [ ]:
# Compare ROC curves
fpr_lr, tpr_lr, _ = roc_curve(y_test_ulb, y_pred_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test_ulb, y_pred_proba_rf)

plt.figure(figsize=(10, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test_ulb, y_pred_proba_lr):.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test_ulb, y_pred_proba_rf):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Baseline Models')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Part 2: Pipeline Simulation with fraudTrain/fraudTest

**⚠️ Important Note:** This section uses a **DIFFERENT dataset** than Part 1:

| Aspect | Part 1 (ULB MLG) | Part 2 (fraudTrain/fraudTest) |
|--------|-------------------|-------------------------------|
| **Dataset** | creditcard.csv | fraudTrain.csv / fraudTest.csv |
| **Features** | PCA components (V1-V28), Time, Amount | Transactional data (amt, lat, long, category, etc.) |
| **Preprocessing** | Scaling only | Feature engineering, encoding, scaling |
| **Purpose** | Baseline model comparison | Real-time pipeline simulation |

These are **completely separate datasets** requiring:
- Different feature sets and schemas
- Different preprocessing pipelines
- Separate models for production deployment

Now we'll transition to simulating a real-time fraud detection pipeline using the fraudTrain and fraudTest datasets.


In [ ]:
# Load fraudTrain and fraudTest datasets
df_train = pd.read_csv('../data/fraudTrain.csv')
df_test = pd.read_csv('../data/fraudTest.csv')

print(f"Training set shape: {df_train.shape}")
print(f"Test set shape: {df_test.shape}")
print(f"\nTraining set columns: {list(df_train.columns)}")
print(f"\nTraining set fraud rate: {df_train['is_fraud'].mean() * 100:.2f}%")
print(f"Test set fraud rate: {df_test['is_fraud'].mean() * 100:.2f}%")


In [ ]:
# Explore the datasets
print("Training set info:")
print(df_train.info())
print("\n" + "="*50)
print("\nTest set info:")
print(df_test.info())


In [ ]:
# Basic statistics
print("Training set statistics:")
print(df_train.describe())
print("\n" + "="*50)
print("\nFraud vs Non-Fraud in training set:")
print(df_train['is_fraud'].value_counts())


### Feature Engineering for Pipeline


In [ ]:
# Convert transaction time to datetime
df_train['trans_date_trans_time'] = pd.to_datetime(df_train['trans_date_trans_time'])
df_test['trans_date_trans_time'] = pd.to_datetime(df_test['trans_date_trans_time'])

# Extract temporal features
def extract_temporal_features(df):
    df = df.copy()
    df['hour'] = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['day_of_month'] = df['trans_date_trans_time'].dt.day
    df['month'] = df['trans_date_trans_time'].dt.month
    return df

df_train = extract_temporal_features(df_train)
df_test = extract_temporal_features(df_test)

print("Temporal features extracted:")
print(df_train[['trans_date_trans_time', 'hour', 'day_of_week', 'day_of_month', 'month']].head())


In [ ]:
# Calculate distance between customer and merchant
def calculate_distance(df):
    df = df.copy()
    # Haversine distance approximation (simplified)
    df['distance'] = np.sqrt(
        (df['lat'] - df['merch_lat'])**2 + (df['long'] - df['merch_long'])**2
    )
    return df

df_train = calculate_distance(df_train)
df_test = calculate_distance(df_test)

print("Distance feature calculated:")
print(df_train[['lat', 'merch_lat', 'long', 'merch_long', 'distance']].head())


In [ ]:
# Prepare features for modeling
# Select numerical features and encoded categorical features
numerical_features = ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 
                      'hour', 'day_of_week', 'day_of_month', 'month', 'distance']

# Encode categorical features
from sklearn.preprocessing import LabelEncoder

categorical_features = ['category', 'gender', 'state']
label_encoders = {}

for feature in categorical_features:
    le = LabelEncoder()
    df_train[f'{feature}_encoded'] = le.fit_transform(df_train[feature].astype(str))
    df_test[f'{feature}_encoded'] = le.transform(df_test[feature].astype(str))
    label_encoders[feature] = le
    numerical_features.append(f'{feature}_encoded')

# Prepare feature matrix
X_train_pipeline = df_train[numerical_features].copy()
X_test_pipeline = df_test[numerical_features].copy()
y_train_pipeline = df_train['is_fraud'].copy()
y_test_pipeline = df_test['is_fraud'].copy()

print(f"Pipeline feature matrix shape: {X_train_pipeline.shape}")
print(f"Features: {list(X_train_pipeline.columns)}")


In [ ]:
# Scale features for pipeline
scaler_pipeline = RobustScaler()
X_train_pipeline_scaled = scaler_pipeline.fit_transform(X_train_pipeline)
X_test_pipeline_scaled = scaler_pipeline.transform(X_test_pipeline)

# Convert to DataFrame
X_train_pipeline_scaled = pd.DataFrame(X_train_pipeline_scaled, columns=X_train_pipeline.columns)
X_test_pipeline_scaled = pd.DataFrame(X_test_pipeline_scaled, columns=X_test_pipeline.columns)

print("Features scaled for pipeline modeling")


### Train Models for Pipeline


In [ ]:
# Train Logistic Regression for pipeline
lr_pipeline = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_pipeline.fit(X_train_pipeline_scaled, y_train_pipeline)

# Predictions
y_pred_lr_pipeline = lr_pipeline.predict(X_test_pipeline_scaled)
y_pred_proba_lr_pipeline = lr_pipeline.predict_proba(X_test_pipeline_scaled)[:, 1]

# Evaluation
print("Pipeline Logistic Regression Results:")
print(f"ROC-AUC Score: {roc_auc_score(y_test_pipeline, y_pred_proba_lr_pipeline):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_pipeline, y_pred_lr_pipeline))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_pipeline, y_pred_lr_pipeline))


In [ ]:
# Train Random Forest for pipeline
rf_pipeline = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
rf_pipeline.fit(X_train_pipeline_scaled, y_train_pipeline)

# Predictions
y_pred_rf_pipeline = rf_pipeline.predict(X_test_pipeline_scaled)
y_pred_proba_rf_pipeline = rf_pipeline.predict_proba(X_test_pipeline_scaled)[:, 1]

# Evaluation
print("Pipeline Random Forest Results:")
print(f"ROC-AUC Score: {roc_auc_score(y_test_pipeline, y_pred_proba_rf_pipeline):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_pipeline, y_pred_rf_pipeline))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_pipeline, y_pred_rf_pipeline))


### Pipeline Simulation: Real-time Scoring


In [ ]:
# Simulate real-time scoring on test set
# Process transactions one by one (simulating streaming)

def simulate_realtime_scoring(model, scaler, df_test_data, feature_cols, label_encoders):
    """
    Simulate real-time fraud detection scoring
    """
    results = []
    
    for idx, row in df_test_data.iterrows():
        # Extract and transform features (as would happen in real-time)
        features = {}
        
        # Numerical features
        features['amt'] = row['amt']
        features['lat'] = row['lat']
        features['long'] = row['long']
        features['city_pop'] = row['city_pop']
        features['merch_lat'] = row['merch_lat']
        features['merch_long'] = row['merch_long']
        
        # Temporal features
        trans_time = pd.to_datetime(row['trans_date_trans_time'])
        features['hour'] = trans_time.hour
        features['day_of_week'] = trans_time.dayofweek
        features['day_of_month'] = trans_time.day
        features['month'] = trans_time.month
        
        # Distance
        features['distance'] = np.sqrt(
            (row['lat'] - row['merch_lat'])**2 + (row['long'] - row['merch_long'])**2
        )
        
        # Categorical encoding
        for cat_feature in ['category', 'gender', 'state']:
            le = label_encoders[cat_feature]
            try:
                features[f'{cat_feature}_encoded'] = le.transform([str(row[cat_feature])])[0]
            except ValueError:
                # Handle unseen categories
                features[f'{cat_feature}_encoded'] = 0
        
        # Create feature vector
        feature_vector = pd.DataFrame([features])[feature_cols]
        
        # Scale
        feature_vector_scaled = scaler.transform(feature_vector)
        
        # Predict
        fraud_probability = model.predict_proba(feature_vector_scaled)[0, 1]
        fraud_prediction = model.predict(feature_vector_scaled)[0]
        
        results.append({
            'transaction_id': idx,
            'fraud_probability': fraud_probability,
            'fraud_prediction': fraud_prediction,
            'actual_fraud': row['is_fraud']
        })
        
        # Print progress every 1000 transactions
        if (idx + 1) % 1000 == 0:
            print(f"Processed {idx + 1} transactions...")
    
    return pd.DataFrame(results)

# Run simulation (on a sample for demonstration)
print("Simulating real-time scoring on test set...")
sample_size = min(5000, len(df_test))  # Process first 5000 transactions
df_test_sample = df_test.head(sample_size).copy()

scoring_results = simulate_realtime_scoring(
    rf_pipeline, 
    scaler_pipeline, 
    df_test_sample,
    numerical_features,
    label_encoders
)

print(f"\nSimulation complete! Processed {len(scoring_results)} transactions.")


In [ ]:
# Analyze simulation results
print("Simulation Results Summary:")
print(f"Total transactions processed: {len(scoring_results)}")
print(f"Fraud detected: {scoring_results['fraud_prediction'].sum()}")
print(f"Actual fraud cases: {scoring_results['actual_fraud'].sum()}")
print(f"\nAccuracy: {(scoring_results['fraud_prediction'] == scoring_results['actual_fraud']).mean():.4f}")

# Confusion matrix for simulation
print("\nConfusion Matrix (Simulation):")
print(confusion_matrix(scoring_results['actual_fraud'], scoring_results['fraud_prediction']))

# Distribution of fraud probabilities
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
scoring_results['fraud_probability'].hist(bins=50, alpha=0.7)
plt.xlabel('Fraud Probability')
plt.ylabel('Frequency')
plt.title('Distribution of Fraud Probabilities')
plt.axvline(x=0.5, color='r', linestyle='--', label='Threshold (0.5)')
plt.legend()

plt.subplot(1, 2, 2)
scoring_results.boxplot(column='fraud_probability', by='actual_fraud', ax=plt.gca())
plt.xlabel('Actual Fraud')
plt.ylabel('Fraud Probability')
plt.title('Fraud Probability by Actual Label')
plt.suptitle('')

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate different thresholds for fraud detection
thresholds = np.arange(0.1, 0.9, 0.05)
precision_scores = []
recall_scores = []
f1_scores = []

for threshold in thresholds:
    predictions = (scoring_results['fraud_probability'] >= threshold).astype(int)
    precision = (scoring_results['actual_fraud'] & predictions).sum() / (predictions.sum() + 1e-10)
    recall = (scoring_results['actual_fraud'] & predictions).sum() / (scoring_results['actual_fraud'].sum() + 1e-10)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
    
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

# Plot threshold analysis
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precision_scores, label='Precision', marker='o')
plt.plot(thresholds, recall_scores, label='Recall', marker='s')
plt.plot(thresholds, f1_scores, label='F1 Score', marker='^')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision, Recall, and F1 Score vs Threshold')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Find optimal threshold (maximizing F1)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
print(f"\nOptimal threshold (F1): {optimal_threshold:.3f}")
print(f"Precision at optimal: {precision_scores[optimal_idx]:.4f}")
print(f"Recall at optimal: {recall_scores[optimal_idx]:.4f}")
print(f"F1 Score at optimal: {f1_scores[optimal_idx]:.4f}")


### Model Comparison Summary


In [ ]:
# Compare all models
comparison_results = {
    'Dataset': ['ULB MLG', 'ULB MLG', 'Pipeline', 'Pipeline'],
    'Model': ['Logistic Regression', 'Random Forest', 'Logistic Regression', 'Random Forest'],
    'ROC-AUC': [
        roc_auc_score(y_test_ulb, y_pred_proba_lr),
        roc_auc_score(y_test_ulb, y_pred_proba_rf),
        roc_auc_score(y_test_pipeline, y_pred_proba_lr_pipeline),
        roc_auc_score(y_test_pipeline, y_pred_proba_rf_pipeline)
    ]
}

comparison_df = pd.DataFrame(comparison_results)
print("Model Comparison:")
print(comparison_df.to_string(index=False))

# Visualize comparison
plt.figure(figsize=(10, 6))
x = np.arange(len(comparison_df))
width = 0.35

plt.bar(x - width/2, comparison_df[comparison_df['Model'] == 'Logistic Regression']['ROC-AUC'], 
        width, label='Logistic Regression', alpha=0.8)
plt.bar(x + width/2, comparison_df[comparison_df['Model'] == 'Random Forest']['ROC-AUC'], 
        width, label='Random Forest', alpha=0.8)

plt.xlabel('Dataset')
plt.ylabel('ROC-AUC Score')
plt.title('Model Performance Comparison')
plt.xticks(x, ['ULB MLG', 'Pipeline'])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Next Steps

1. **Save trained models** for use in production pipeline
2. **Implement streaming pipeline** using stream_simulator.py
3. **Add monitoring** for model drift and performance metrics
4. **Deploy scoring service** for real-time fraud detection


In [ ]:
# Save models and preprocessors for production use
import pickle
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save ULB MLG models
with open('../models/lr_ulb_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
    
with open('../models/rf_ulb_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
    
with open('../models/scaler_ulb.pkl', 'wb') as f:
    pickle.dump(scaler_ulb, f)

# Save Pipeline models
with open('../models/lr_pipeline_model.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)
    
with open('../models/rf_pipeline_model.pkl', 'wb') as f:
    pickle.dump(rf_pipeline, f)
    
with open('../models/scaler_pipeline.pkl', 'wb') as f:
    pickle.dump(scaler_pipeline, f)
    
with open('../models/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

print("Models and preprocessors saved successfully!")
